# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Print dataset metadata overview
meta = dataset.metadata
print('Dataset loaded!')
print(f"Name: {meta.name}")
print(f"Description: {meta.description}")
print(f"Version: {getattr(meta, 'version', None)}")
print(f"License: {getattr(meta, 'license', None)}")
if getattr(meta, 'keywords', []) != []:
    print(f"Keywords: {meta.keywords}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields using @id
if hasattr(dataset, 'record_sets'):
    print(f"Number of record sets: {len(dataset.record_sets)}\n")
    for i, rs in enumerate(dataset.record_sets):
        print(f"Record Set {i}: @id = {rs.id}")
        print(f"  Name: {rs.name}")
        print(f"  Description: {getattr(rs, 'description', '')}")
        if hasattr(rs, 'fields') and rs.fields:
            for f in rs.fields:
                print(f"    Field @id: {f.id} | name: {f.name} | type: {f.data_type}")
        print()
else:
    # Try top-level fields as fallback (older Croissant schemas)
    print("No explicit record sets found in the schema.")
    # Sometimes a single default record set is available
    try:
        rs = dataset.record_set
        print(f"Default Record Set: @id = {rs.id} | name: {rs.name}")
        if hasattr(rs, 'fields') and rs.fields:
            for f in rs.fields:
                print(f"    Field @id: {f.id} | name: {f.name} | type: {f.data_type}")
    except Exception as e:
        print(str(e))

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set by @id
import warnings
warnings.filterwarnings("ignore")

# Get all record set IDs
try:
    record_sets = dataset.record_sets
    record_set_ids = [rs.id for rs in record_sets]
except AttributeError:
    # Fallback for schemas with single record set
    record_sets = [dataset.record_set]
    record_set_ids = [dataset.record_set.id]

dataframes = {}

for rs_id in record_set_ids:
    # Load up to 10,000 records per set for demonstration
    records = list(dataset.records(record_set=rs_id))
    if len(records) == 0:
        print(f"No records for record set {rs_id}")
        continue
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[rs_id])} records from record set '{rs_id}'\nColumns:")
    print(dataframes[rs_id].columns.tolist())
    # Preview
    display(dataframes[rs_id].head())
    # For demonstration, only show one set
    break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Pick the first loaded record set for EDA
if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Find a numeric field for demo (e.g. columns containing "count", "coverage", "MW")
    numeric_columns = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int] or pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_columns:
        # Try to coerce columns that look numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_columns.append(col)
            except Exception:
                pass

    print(f"Numeric fields found: {numeric_columns}")
    if numeric_columns:
        numeric_field = numeric_columns[0]
        threshold = df[numeric_field].quantile(0.9)    # Use 90th percentile for threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with '{numeric_field}' > {threshold} (top 10%):")
        display(filtered_df.head())

        # Normalizing
        filtered_df = filtered_df.copy()  # Just in case of warnings
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by something categorical if available
        group_field = None
        for c in df.columns:
            if df[c].dtype == object and df[c].nunique() < 10 and c != numeric_field:
                group_field = c
                break
        if group_field is not None:
            # Use mean of normalized in group
            grouped_df = filtered_df.groupby(group_field)[f"{numeric_field}_normalized"].mean().reset_index()
            print(f"Grouped (mean normalized {numeric_field}) by '{group_field}':")
            display(grouped_df)
    else:
        print("No numeric fields for EDA in this record set.")
else:
    print("No dataframes loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the main numeric field filtered above
if len(dataframes) > 0 and 'numeric_field' in locals():
    # Histogram of all values
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of field: {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouped_df exists, show bar plot
    if 'grouped_df' in locals() and group_field is not None:
        plt.figure(figsize=(8,4))
        sns.barplot(data=grouped_df, x=group_field, y=f"{numeric_field}_normalized")
        plt.title(f"Mean Normalized {numeric_field} per {group_field}")
        plt.ylabel(f"Mean Normalized {numeric_field}")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Croissant dataset schema enabled discovery of available record sets and fields entirely via `@id` references.
- Data can be loaded, filtered on numeric fields, normalized, and grouped without explicit knowledge of the physical data format.
- The main numeric attributes (such as counts, molecular weights, or coverages) provide a useful basis for EDA and visualization in mass spectrometry protein datasets.
- For further analysis, refer to specific documentation sections referenced in the Croissant schema for ontology and field meaning.